# Project Management Data Analysis

This notebook explores a synthetic dataset of project management metrics and builds predictive models to classify project outcomes. The dataset includes information such as project type, duration, budget, team size, risk scores, stakeholder satisfaction, and whether the project was successful.

## Objectives
1. Perform exploratory data analysis (EDA) to understand patterns in the data.
2. Visualize relationships between key variables.
3. Build and evaluate predictive models to classify project outcomes.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# Set display options for pandas
pd.set_option('display.max_columns', None)


In [ ]:
# Load dataset
data = pd.read_csv('project_data.csv', parse_dates=['Start_Date','End_Date'])

# Preview the data
data.head()

In [ ]:
# Basic statistics
print('Dataset shape:', data.shape)
print('
Data types:
', data.dtypes)
print('
Summary statistics:')
data.describe(include='all')

In [ ]:
# Distribution of numeric variables
plt.figure(figsize=(12, 8))
for i, col in enumerate(['Duration_Days', 'Budget_KUSD', 'Actual_Cost_KUSD', 'Team_Size', 'Risk_Score', 'Stakeholder_Satisfaction']):
    plt.subplot(2, 3, i+1)
    sns.histplot(data[col], kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
numeric_cols = ['Duration_Days','Budget_KUSD','Actual_Cost_KUSD','Team_Size','Risk_Score','Stakeholder_Satisfaction','Outcome']
correlation = data[numeric_cols].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Prepare features and target
X = data.drop(['Project_ID','Start_Date','End_Date','Outcome'], axis=1)
y = data['Outcome']

# Identify categorical and numerical columns
cat_cols = ['Project_Type']
num_cols = [col for col in X.columns if col not in cat_cols]

# Preprocessing: one-hot encode categorical variables and scale numeric variables if desired
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# Logistic Regression model
log_reg_pipeline = Pipeline(steps=[('preprocess', preprocessor),
                                  ('classifier', LogisticRegression(max_iter=1000))])

# Train
log_reg_pipeline.fit(X_train, y_train)

# Predict
y_pred_lr = log_reg_pipeline.predict(X_test)

# Evaluate
print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_lr))
print('
Classification Report (Logistic Regression):
', classification_report(y_test, y_pred_lr))
print('Confusion Matrix (Logistic Regression):
', confusion_matrix(y_test, y_pred_lr))


In [ ]:
# Random Forest model
rf_pipeline = Pipeline(steps=[('preprocess', preprocessor),
                              ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))])

# Train
rf_pipeline.fit(X_train, y_train)

# Predict
y_pred_rf = rf_pipeline.predict(X_test)

# Evaluate
print('Random Forest Accuracy:', accuracy_score(y_test, y_pred_rf))
print('
Classification Report (Random Forest):
', classification_report(y_test, y_pred_rf))
print('Confusion Matrix (Random Forest):
', confusion_matrix(y_test, y_pred_rf))
